In [1]:
import os
from pathlib import Path
import shutil
from collections import defaultdict

from PIL import Image  # pip install pillow se manca
from python_file.dirPath import csvDir, txtDir, imageDir, dayTest, fogTest, nightTest, Test

In [2]:
def remap_dataset(src_dirs, dst_dir, offsets, image_ext=".png", label_file="prova.txt"):
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    merged_lines = []

    for src_dir, offset in zip(src_dirs, offsets):
        src_dir = Path(src_dir)
        txt_path = src_dir / label_file

        mapping = {}

        for img_path in sorted(src_dir.glob(f"*{image_ext}")):
            old_name = img_path.name
            old_stem = img_path.stem

            if not old_stem.isdigit():
                continue

            new_idx = int(old_stem) + offset
            new_name = f"{new_idx}{image_ext}"
            mapping[old_name] = new_name

            shutil.copy2(img_path, dst_dir / new_name)

        if txt_path.exists():
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue

                    old_img = parts[0]
                    if old_img in mapping:
                        parts[0] = mapping[old_img]
                        merged_lines.append(" ".join(parts))

    with open(dst_dir / label_file, "w", encoding="utf-8") as f:
        for line in merged_lines:
            f.write(line + "\n")

In [3]:
src_dirs = [
    dayTest,
    nightTest,
    fogTest
]

offsets = [0, 1000, 2000]

remap_dataset(
    src_dirs=src_dirs,
    dst_dir=Test,
    offsets=offsets
)